In [29]:
import pandas as pd
import matplotlib.pyplot as plt

In [30]:
data = pd.read_csv("train.txt" , sep= ';' , header = None , names = ['text' , 'emotion' ] )

In [31]:
data

,text,emotion
0,i didnt feel humiliated,sadness
1,i can go from feeling so hopeless to so damned...,sadness
2,im grabbing a minute to post i feel greedy wrong,anger
3,i am ever feeling nostalgic about the fireplac...,love
4,i am feeling grouchy,anger
...,...,...
15995,i just had a very brief time in the beanbag an...,sadness
15996,i am now turning and i feel pathetic that i am...,sadness
15997,i feel strong and good overall,joy
15998,i feel like this was such a rude comment and i...,anger


In [32]:
data["emotion"].unique()

array(['sadness', 'anger', 'love', 'surprise', 'fear', 'joy'],
      dtype=object)

In [33]:
data = data.replace({'sadness':0, 'anger':1, 'love' :2 , 'surprise' : 3, 'fear': 4, 'joy': 5 })

/tmp/ipykernel_2648/53361653.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data = data.replace({'sadness':0, 'anger':1, 'love' :2 , 'surprise' : 3, 'fear': 4, 'joy': 5 })


In [34]:
data["text"] = data["text"].str.lower()

In [35]:
data["text"] = data["text"].str.replace(r'[^\w\s]',' ',regex=True)

In [36]:
data["text"] = data["text"].str.replace(r'\d+',' ',regex=True)


In [37]:
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize


In [38]:
import nltk
nltk.download('stopwords')
nltk.download('punkt_tab')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [39]:
stop= stopwords.words("english")

In [40]:
data_tokenize = word_tokenize(str(data["text"]))

In [41]:
def remove(text):
    text = str(text)
    words=word_tokenize(text)

    cleaned= [i for i in words if i.lower() not in stop]
    return' '.join(cleaned)

In [42]:
data["text"] = data["text"].apply(remove)

In [43]:
data["text"]

,text
0,didnt feel humiliated
1,go feeling hopeless damned hopeful around some...
2,im grabbing minute post feel greedy wrong
3,ever feeling nostalgic fireplace know still pr...
4,feeling grouchy
...,...
15995,brief time beanbag said anna feel like beaten
15996,turning feel pathetic still waiting tables sub...
15997,feel strong good overall
15998,feel like rude comment im glad


In [44]:
x= data["text"]
y=data["emotion"]

In [45]:
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(
x, y, test_size=0.2, random_state=42)

In [46]:
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer

In [47]:
bow_vect = CountVectorizer()

In [48]:
x_train_bow = bow_vect.fit_transform(x_train)
x_test_bow = bow_vect.transform(x_test)

In [49]:
from sklearn.naive_bayes import MultinomialNB
NB= MultinomialNB()

In [50]:
NB.fit(x_train_bow,y_train)

MultinomialNB()

In [51]:
pred = NB.predict(x_test_bow)

In [52]:
from sklearn.metrics import accuracy_score , recall_score,f1_score

In [53]:
print(accuracy_score(y_test,pred))


0.7678125


In [54]:
tfid = TfidfVectorizer()
x_train_tfid = tfid.fit_transform(x_train)
x_test_tfid = tfid.transform(x_test)

In [55]:
NB2= MultinomialNB()
NB2.fit(x_train_tfid,y_train)

MultinomialNB()

In [56]:
pred = NB.predict(x_test_tfid)

In [57]:
print(accuracy_score(y_test,pred))

0.7275


In [58]:
from sklearn.linear_model import LogisticRegression
LR = LogisticRegression(max_iter =500 )


In [59]:
LR.fit(x_train_tfid,y_train)

LogisticRegression(max_iter=500)

In [60]:
pred = LR.predict(x_test_tfid)

In [61]:
print(accuracy_score(y_test,pred))

0.8615625


### Preparing Data for RNN (Tokenization, Embedding, and Padding)



In [69]:
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Hyperparameters
vocab_size = 10000
max_length = 100
embedding_dim = 64

# Initialize and fit the tokenizer
tokenizer = Tokenizer(num_words=vocab_size, oov_token='<OOV>')
tokenizer.fit_on_texts(x_train)

# Convert texts to sequences
x_train_seq = tokenizer.texts_to_sequences(x_train)
x_test_seq = tokenizer.texts_to_sequences(x_test)

# Pad sequences using 'pre' padding to preserve context at the end
x_train_pad = pad_sequences(x_train_seq, maxlen=max_length, padding='pre', truncating='pre')
x_test_pad = pad_sequences(x_test_seq, maxlen=max_length, padding='pre', truncating='pre')

print(f"Training shape: {x_train_pad.shape}")
print(f"Testing shape: {x_test_pad.shape}")

Training shape: (12800, 100)
Testing shape: (3200, 100)


### Build and Train the RNN Model



In [66]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

num_classes = len(y.unique())

#the LSTM model
model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=embedding_dim),
    LSTM(units=64, dropout=0.2, recurrent_dropout=0.2),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(num_classes, activation='softmax')
])
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [70]:
history = model.fit(
    x_train_pad, y_train,
    epochs=10,
    validation_data=(x_test_pad, y_test),
    batch_size=64
)

Epoch 1/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 24s 119ms/step - accuracy: 0.4265 - loss: 1.3044 - val_accuracy: 0.6963 - val_loss: 0.8223
Epoch 2/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 40s 116ms/step - accuracy: 0.8382 - loss: 0.4821 - val_accuracy: 0.9053 - val_loss: 0.3183
Epoch 3/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 23s 117ms/step - accuracy: 0.9405 - loss: 0.1821 - val_accuracy: 0.9156 - val_loss: 0.2770
Epoch 4/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 24s 118ms/step - accuracy: 0.9634 - loss: 0.1117 - val_accuracy: 0.9191 - val_loss: 0.2524
Epoch 5/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 23s 116ms/step - accuracy: 0.9710 - loss: 0.0847 - val_accuracy: 0.9225 - val_loss: 0.2593
Epoch 6/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 22s 111ms/step - accuracy: 0.9790 - loss: 0.0630 - val_accuracy: 0.9206 - val_loss: 0.2748
Epoch 7/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 42s 119ms/step - accuracy: 0.9822 - loss: 0.0522 - val_accuracy: 0.9231 - val_loss: 0.2816
Epoch 8/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 41s 120ms/step - accuracy: 0.9854 - loss: 0

In [71]:
loss, accuracy = model.evaluate(x_test_pad, y_test, verbose=0)
print(f"LSTM Test Accuracy: {accuracy:.4f}")

LSTM Test Accuracy: 0.9153


In [74]:
import joblib
import tensorflow as tf

tf.keras.models.save_model(model, "analysis_model.keras")
joblib.dump(tokenizer, "tokenizer.pkl")

['tokenizer.pkl']